## 1-minute introduction to Jupyter ##

A Jupyter notebook consists of cells. Each cell contains either text or code.

A text cell will not have any text to the left of the cell. A code cell has `In [ ]:` to the left of the cell.

If the cell contains code, you can edit it. Press <kbd>Enter</kbd> to edit the selected cell. While editing the code, press <kbd>Enter</kbd> to create a new line, or <kbd>Shift</kbd>+<kbd>Enter</kbd> to run the code. If you are not editing the code, select a cell and press <kbd>Ctrl</kbd>+<kbd>Enter</kbd> to run the code.

---

# Object-Oriented Programming

This lesson continues from the previous chapter on Inheritance. In that lesson, we created `TrackingBoard` and `PlayingBoard` classes that inherit from `Grid`, allowing them to be used wherever `Grid` is required, yet also adding additional functionality. This avoids code duplication and encourages code reuse.

Now we turn our attention to the player. In the starting code, our game loop looked like this:

```python
    while not is_gameover(turns, player_hits, enemy_hits, total_ship_cells):
        print("\nTurns left:", turns)
        print("\nPlayer's turn")
        display_board(player_targetting)
        x, y = prompt_valid_guess(player_targetting)

        hit_char = enemy_board[x][y]
        targetting_update(player_targetting, hit_char, x, y)
        if is_target_hit(enemy_board, x, y):
            player_hits += 1
            enemy_ship_cells[hit_char] -= 1
            if enemy_ship_cells[hit_char] == 0:
                print("You sunk the", hit_char + "!")
                player_sunk_ships.append(hit_char)
        print("Ships sunk:", " ".join(player_sunk_ships) or "None")

        print("\nEnemy's turn")
        x, y = get_enemy_guess(player_board)

        hit_char = player_board[x][y]
        targetting_update(enemy_targetting, hit_char, x, y)
        if is_target_hit(player_board, x, y):
            enemy_hits += 1
            player_ship_cells[hit_char] -= 1
            if player_ship_cells[hit_char] == 0:
                print("Enemy sunk the", hit_char + "!")
                enemy_sunk_ships.append(hit_char)
        display_overlay(enemy_targetting, player_board)
        print("Ships sunk:", " ".join(enemy_sunk_ships) or "None")
        turns -= 1
```

This code is tricky to abstract, because the player and enemy must be handled differently. The player must be prompted for input, while the enemy's guess is generated randomly. The player also has a different board to display than the enemy.

We can abstract this code by creating a `Player` class that handles the player's input and a `Computer` class that handles the computer's input. This allows us to create a single game loop that works for both players and computers.

What if we could rewrite the code like this?


In [ ]:
# Copy your TargettingBoard and PlayingBoard class from the previous exercise into this cell
# and run this cell so it is available for the example code cell below.



In [ ]:
from starting_code import GRID_SIZE, EMPTY, HIT, MISS

class Commander:
    """Superclass for Player and Computer.

    Subclasses must implement the guess() method.
    
    Methods:
        guess: Returns a tuple of integers representing the coordinates of the guess.
    """
    def __init__(self, name):
        self.name = name
        self.targetting = TargettingBoard(GRID_SIZE, EMPTY)
        self.playing = PlayingBoard(GRID_SIZE, EMPTY)

    def guess(self) -> tuple[int, int]:
        """Returns a tuple of integers representing the coordinates of the guess."""
        raise NotImplementedError("Subclasses must implement this method.")

class Player(Commander):
    """We will implement guess() later"""

class Computer(Commander):
    """We will implement guess() later"""
    

def execute_turn(attacker: Commander, defender: Commander) -> None:
    """Executes a turn for each commander.
    
    A turn comprises:
    1. The attacker guesses a coordinate.
    2. The defender checks if the guess hits a ship.
    3. The attacker updates the targetting board with the result of the guess.
    4. The defender updates the playing board with the result of the guess.
    5. If the guess hits a ship, the defender checks if the ship is sunk.
    6. If the ship is sunk, the attacker is notified.
    7. The attacker and defender boards are displayed.

    Args:
        attacker (Commander): The commander who is attacking.
        defender (Commander): The commander who is defending.
    """
    # Attacker's turn
    print(f"\n{attacker.name}'s turn")
    attacker.targetting.display()
    x, y = attacker.guess()

    hit_char = defender.playing.get(x, y)
    attacker.targetting.update(x, y, hit_char)  # also updates attacker.targetting.hits
    defender.playing.update(x, y)  # updates ship_cells and sunk_ships
    if defender.is_sunk(hit_char):  # assuming the existence of this method; it can be implemented if not existent
        print(f"{attacker} sunk the", hit_char + "!")
    # Only the player needs to see the overlay
    if isinstance(attacker, Player):
        display_overlay(attacker.targetting, defender.board)

def run_game(turns: int, ships: dict[str, int]) -> None:
    total_ship_cells = sum(ships.values())
    player = Player("You")
    enemy = Computer("Enemy")
    # Place ships on the boards
    for board in [player.playing, enemy.playing]:
        for name, size in ships.items():
            board.place_ship(name, size)
            # ship_cells are updated within the place_ship() method

    while not is_gameover(turns: int, player: Player, enemy: Computer, total_ship_cells: int):
        print("\nTurns left:", turns)
        # Use two execute_turn() function calls to execute a turn for each commander
        # by swapping the attacker and defender
        execute_turn(player, enemy)
        execute_turn(enemy, player)
        turns -= 1

    # Game is over
    if is_won(player, total_ship_cells):
        print("Congratulations! You sank all the enemy ships!")
        display_overlay(player, enemy)
    elif is_won(enemy, total_ship_cells):
        print("Game over! The enemy sank all your ships!")
        display_overlay(enemy, player)
    else:
        print("Game over! You ran out of turns!")
        display_overlay(player, enemy)


## Polymorphism

This pattern enables us to compress the game loop significantly; we avoiding writing lots of if-else statements to handle the player and computer differently, and we also avoid code duplication for the player and computer. The tradeoff is that the player and computer must **share a common interface**. For `execute_turn()` to work, both `Player` and `Computer` must have a public `name` attribute and a public `guess()` method. We can use a superclass to describe this interface, and implement any common functionality so that it does not need to be duplicated in the subclasses.

`Player.guess()` will use the `input()` function to prompt the player for input, validating it and converting it into a tuple, while `Computer.guess()` will generate a random valid guess. Although both classes obey the same interface, they are implemented differently. This is the essence of the principle of **polymorphism**: different classes can be used interchangeably as long as they share a common interface.

## Exercise 1

Implement the `Player` and `Computer` classes. The `Player` class should have a `name` attribute and a `guess()` method that prompts the player for input. The `Computer` class should have a `name` attribute and a `guess()` method that generates a random valid guess.

You may use the starting code, refactoring it to achieve the above requirements.

In [ ]:
# Write your code below

## Exercise 2

In a separate file, `battleships.py`, rewrite the starting code, applying the principles of encapsulation, inheritance, and polymorphism.

Your code should have the following classes:

- `Grid`: The base class for `TrackingBoard` and `PlayingBoard`.
- `TrackingBoard`: A subclass of `Grid` that tracks hits and misses.
- `PlayingBoard`: A subclass of `Grid` that tracks ship positions.
- `Commander`: The base class for `Player` and `Computer`.
- `Player`: A class that represents the player.
- `Computer`: A class that represents the computer.

Introduce any necessary helper functions you require, and remove any unnecessary functions. You may need to refactor some functions from the starting code to make them work with the new classes.

## Exercise 3 (optional)

In a separate `game.py` file, encapsulate the game loop in a `BattleshipsGame` class. The `BattleshipsGame` class should have the following public interface:

- `__init__(self, player_name: str)`: Initialises the game and any attributes required by the class
- `play(self)`: Starts the game loop
- `is_gameover(self)`: Checks if the game is over

(You may introduce any other helper methods, but they should be private.)

The game should be playable with the following code example:

```python
from battleships import BattleshipsGame

game = BattleshipsGame("You")
game.play()
```